In [1]:
# pip install --upgrade sendgrid

In [2]:
# pip install --upgrade cryptography

In [3]:
# pip install --user openai-agents

In [19]:
import os
import asyncio
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content
from dotenv import load_dotenv

In [5]:
from agents import Agent, Runner, trace, function_tool

In [15]:
from openai.types.responses import ResponseTextDeltaEvent

In [6]:
load_dotenv(override=True)

True

In [17]:
def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.getenv("SENDGRID_API_KEY"))
    from_email = Email("wetechfin@gmail.com")
    to_email = To("debmalyamondal63@gmail.com")
    content = Content("text/plain", "This is a the test email body")
    mail = Mail(from_email, to_email, "test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

In [19]:
send_test_email()

202


In [8]:
instruction1 = "You are a sales agent working for AutAI, \
an AI automation agency company that provides automated solution for small and medium business. \
You write professional, serious cold emails"

instruction2 = "You are a humorous, engaging sales agent working for AutAI, \
an AI automation agency company that provides automated solution for small and medium businesses. \
You write witty, engaging cold emails that are likely to get a response."

instruction3 = "You are a busy sales agent working for AutAI, \
an AI automation agency company that provides automated solution for small and medium businesses. \
You write concise, to the point cold emails"



In [9]:
sales_agent1 = Agent(
    name="Professional sales agent",
    instructions=instruction1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="Engaging sales agent",
    instructions=instruction2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Busy sales agent",
    instructions=instruction3,
    model="gpt-4o-mini"
)


In [17]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, flush=True, end="")

Subject: Elevate Your Business Efficiency with Automation

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I represent AutAI, an agency dedicated to helping small and medium businesses streamline their operations through tailored AI automation solutions.

In today’s fast-paced market, spending valuable time on repetitive tasks can hinder your growth. Our team specializes in helping businesses like yours save time and resources by automating processes such as customer support, data management, and sales workflows. 

I would love the opportunity to discuss how we can create a custom solution that meets your unique needs and drives measurable improvements in efficiency. 

Would you be available for a brief call next week? 

Thank you for considering this opportunity. I look forward to the possibility of working together to boost your business operations.

Best regards,

[Your Name]  
[Your Title]  
AutAI  
[Your Phone Number]  
[Your Email]  
[Web

In [25]:
message = "Write a cold email"

with trace("Parellel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )
    outputs = [result.final_output for result in results]
    
    emails = "cold sales emails: \n\n" + "\n\nEmail: \n\n".join(outputs)

Subject: Unlock Your Business Potential with AI Automation

Hi [Recipient's Name],

I hope this message finds you well.

My name is [Your Name], and I am with AutAI, an agency dedicated to helping small and medium businesses like yours leverage the power of AI automation. In today’s fast-paced marketplace, efficiency and innovation are crucial to staying competitive.

We specialize in tailoring automation solutions that streamline operations, reduce costs, and enhance overall productivity. Our clients have reported significant improvements in workflow efficiency and a noticeable increase in revenue after implementing our solutions.

I would love to discuss how we can help your business achieve similar results. Are you available for a brief call next week to explore the possibilities?

Thank you for your time, and I look forward to the opportunity to connect.

Best regards,

[Your Name]  
[Your Position]  
AutAI  
[Your Phone Number]  
[Your Email Address]  
[Your Website]  
Subject: Ti